# Homework 1 - Information Retrieval.


Jeroen Baars - 10686320

Bob van den Hoogen - 10420169

Lea .

## Module 1: Evaluation.

Deadline: Wednesday, January 17th, midnight (23:59).

Collaboration: This is a team-based assignment. Form teams of 3 people.

Submit: An IPython Notebook for both the theoretical and the experimental parts with the necessary (a) answers, (b) implementation, (c) explanations,  (d) comments, and (e) analysis. Code quality, informative comments, detailed explanations of what each block in the notebook does and more important convincing analysis of the results will be considered when grading. 

Submit your work through Blackboard. All students in a team need to submit their work.

Filename: < student 1 id >–< student 2 id >–< student 3 id >–hw1.ipynb.

The homework will cover the following three topics covered in Lecture 1, 2, and 3:
- Evaluation measures;
- Interleaving;
- Click models.

The homework has a small theoretical part and a large experimental part.


## Theoretical Part [15 pts]

__Question 1.Hypothesis Testing – The problem of multiple comparisons [5 points]
Experimentation in AI often happens like this:__
- Modify/Build an algorithm
- Compare the algorithm to a baseline by running a hypothesis test.
- If not significant, go back to step A
- If significant, start writing a paper. 

Compute the probabilities below ~~How many hypothesis tests, m, does it take to get to~~ (with Type I error for each test = α):

__A. P(mth experiment gives significant result | m experiments lacking power to reject H0)?__

Answer:
    
mogelijk iets:
https://www.stat.berkeley.edu/~mgoldman/Section0402.pdf

piazza staat de vraag anders geformuleerd en uitgelegd:

The chance of getting a Type I error is present in all experiments and independently it is $\alpha$ We care about the m-th experiment making an actual mistake. Here is a reformulation of the question:

 

Suppose hypothetically that the null hypothesis is actually true.

 

The probability of concluding it is false after one test is α. If we do not find it false, we run a second experiment.

What is the probability of concluding that it is false in this second experiment? If we do not find it false, we run a third experiment. And so on and so forth. If we do not find it wrong after m-1 experiment, we run an m-th one. What is the probability of concluding that it is false in this m-th experiment?



__B. P(at least one significant result | m experiments lacking power to reject H0)?__

Answer:


__Question 2. Bias and unfairness in Interleaving experiments [10 points].__

Balance interleaving has been shown to be biased in a number of corner cases. An example was given during the lecture with two ranked lists of length 3 being interleaved, and a randomly clicking population of users that resulted in algorithm A winning ⅔ of the time, even though in theory the percentage of wins should be 50% for both algorithms. Can you come up with a situation of two ranked lists of length 3 and a distribution of clicks over them for which Team-draft interleaving is unfair to the better algorithm?

Answer:




## Experimental Part [85 pts]

In [3]:
import itertools
import numpy as np


Commercial search engines use both offline and online approach in evaluating a new search algorithm: they first use an offline test collection to compare the production algorithm (P) with the new experimental algorithm (E); if E statistically significantly outperforms P with respect to the evaluation measure of their interest, the two algorithms are then compared online through an interleaving experiment.

For the purpose of this homework we will assume that the evaluation measures of interest are:
- Binary evaluation measures
    - Precision at rank k,
    - Recall at rank k,
    - Average Precision,
- Multi-graded evaluation measures
    - Normalized Discounted Cumulative Gain at rank k (nDCG@k),
    - Expected Reciprocal Rank (ERR).
    
Further, for the purpose of this homework we will assume that the interleaving algorithms of interest are:
- Team-Draft Interleaving (Joachims. "Evaluating retrieval performance using clickthrough data". Text Mining 2003.),
- ~~Probabilistic Interleaving (Hofmann, Whiteson, and de Rijke. "A probabilistic method for inferring preferences from clicks." CIKM 2011.).~~
 
In an interleaving experiment the ranked results of P and E (against a user query) are interleaved in a single ranked list which is presented to a user. The user then clicks on the results and the algorithm that receives most of the clicks wins the comparison. The experiment is repeated for a number of times (impressions) and the total wins for P and E are computed. 

A Sign/Binomial Test is then run to examine whether the difference in wins between the two algorithms is statistically significant (or due to chance). Alternatively one can calculate the proportion of times the E wins and test whether this proportion, p, is greater than p0=0.5. This is called an 1-sample 1-sided proportion test.

One of the key questions however is __whether offline evaluation and online evaluation outcomes agree with each other__. In this homework you will determine the degree of agreement between offline evaluation measures and interleaving outcomes, by the means of simulations. A similar analysis using actual online data can be found at Chapelle et al. “Large-Scale Validation and Analysis of Interleaved Search Evaluation”.


### [Based on Lecture 1]

__Step 1: Simulate Rankings of Relevance for E and P (5 points).__

In the first step you will generate pairs of rankings of relevance, for the production P and experimental E, respectively, for a hypothetical query q. Assume a 3-graded relevance, i.e. {N, R, HR}. Construct all possible P and E ranking pairs of length 5.<br>

This step should give you about.<br>
Example:<br>
P: {N N N N N}<br>
E: {N N N N R}<br>
…<br>
P: {HR HR HR HR R}<br>
E: {HR HR HR HR HR}<br>

(Note 1: If you do not have enough computational power, sample 5000 pair uniformly at random to show your work.)


In [4]:
combis = list(itertools.product(['N','R','HR'], repeat=10))
combinations = [[comb[:5],comb[5:]] for comb in combis if comb[:5] != comb[5:]]
for i in combinations[0:3]:
    print(i)
for j in combinations[len(combinations)-3:]:
    print(j)

[('N', 'N', 'N', 'N', 'N'), ('N', 'N', 'N', 'N', 'R')]
[('N', 'N', 'N', 'N', 'N'), ('N', 'N', 'N', 'N', 'HR')]
[('N', 'N', 'N', 'N', 'N'), ('N', 'N', 'N', 'R', 'N')]
[('HR', 'HR', 'HR', 'HR', 'HR'), ('HR', 'HR', 'HR', 'R', 'HR')]
[('HR', 'HR', 'HR', 'HR', 'HR'), ('HR', 'HR', 'HR', 'HR', 'N')]
[('HR', 'HR', 'HR', 'HR', 'HR'), ('HR', 'HR', 'HR', 'HR', 'R')]


__Step 2: Implement Evaluation Measures (10 points).__

Implement 1 binary and 2 multi-graded evaluation measures out of the 7 measures mentioned above. 
(Note 2: Some of the aforementioned measures require the total number of relevant and highly relevant documents in the entire collection – pay extra attention on how to find this)


In [20]:
# precision at rank k

good_p = 0
good_e = 0
for pair in combinations[0:20000]:
    for doc in pair[0][0:3]:
        if doc == 'R' or doc == 'HR':
            good_p +=1
    for doc in pair[1][0:3]:
        if doc == 'R' or doc == 'HR':
            good_e +=1

prec_p = good_p/(len(combinations)*3)
prec_e = good_e/(len(combinations)*3)
print("precision of p is : ", prec_p)
print("precision of e is : ", prec_e)


precision of p is :  0.15040415377115715
precision of e is :  0.22687027400832116


/Users/bobvandenhoogen/anaconda/lib/python3.6/site-packages/ipykernel_launcher.py:38: RuntimeWarning: invalid value encountered in double_scalars
/Users/bobvandenhoogen/anaconda/lib/python3.6/site-packages/ipykernel_launcher.py:48: RuntimeWarning: invalid value encountered in double_scalars


NDCG of p is:  nan


In [43]:
# Normalized Discounted Cumulative Gain at rank k
ranks = {'N': 0, 'R': 1, 'HR': 2}

NDCG_list_p = []
NDCG_list_e = []

for pair in combinations[0:20000]:
    DCG_p = 0
    IDCG_p = 0
    DCG_e = 0
    IDCG_e = 0
    
    ideal_p = []
    for i, doc in enumerate(pair[0][0:3]):      
        CG_p = ranks[doc]
        ideal_p.append(CG_p)
        DCG_p += (2 ** CG_p - 1) / (np.log(i + 2.0)/ np.log(2.0))
    ideal_p.sort(reverse=True)
    for i, CG in enumerate(ideal_p):
        IDCG_p += (2 ** CG - 1) / (np.log(i + 2.0)/ np.log(2.0))
        NDCG_p = 0 if IDCG_p == 0 else DCG_p / IDCG_p
    NDCG_list_p.append(NDCG_p)
        
    ideal_e = []
    for i, doc in enumerate(pair[1][0:3]):      
        CG_e = ranks[doc]
        ideal_e.append(CG_e)
        DCG_e += (2 ** CG_e - 1) / (np.log(i + 2.0)/ np.log(2.0))
    ideal_e.sort(reverse=True)
    for i, CG in enumerate(ideal_e):
        IDCG_e += (2 ** CG - 1) / (np.log(i + 2.0)/ np.log(2.0))
        NDCG_e = 0 if IDCG_e == 0 else DCG_e / IDCG_e
    NDCG_list_e.append(NDCG_e)

print('average NDCG of p is: ', np.average(NDCG_list_p))
print('average NDCG of e is: ', np.average(NDCG_list_e))

average NDCG of p is:  0.552921590498
average NDCG of e is:  0.797606070489


In [ ]:
ERRe = 0
ERRp = 0

PN = 0
PR = 1/4
PHR = 3/4

Pstop = [0,0,0,0,0]
ERR = [0,0,0,0,0]

def check_p(doc):
    if doc == 'N':
        return PN
    elif doc == 'R':
        return PR
    elif doc == 'HR':
        return PHR

ERR = []

for pair in combinations[0:20000]:
    
    ERRsum = [0,0]
    ERRpartial = [0,0,0,0,0]
    
    for index, query in enumerate(pair):
        for i, doc in enumerate(query):
            Pstop[i] = check_p(doc)
        for j, p in enumerate(Pstop):
            for k in range(j):
                if k == 0:
                    sumup = 1-Pstop[k]
                else:
                    sumup *= (1/k)*(1-Pstop[k])
            if j == 0:
                total = Pstop[j]
            else:
                total = (1/(j+1)) * Pstop[j] * sumup
            ERRpartial[j] = total
        ERRsum[index] = sum(ERRpartial)
    ERR.append(ERRsum)

deltaERR = []

for ind, comb in enumerate(ERR):
    if comb[1] > comb[0]:
        deltaERR.append([ind, comb[1]-comb[0]])
        
for comb in deltaERR[0:20]:
    print(comb)

__Step 3: Calculate the 𝛥measure (0 points).__

For the three measures and all P and E ranking pairs constructed above calculate the difference: 𝛥measure = measureE-measureP. Consider only those pairs for which E outperforms P.


In [ ]:
# NCDG
d_NDCG = []
d_NDCG_ix = []
for i in range(len(NDCG_list_e)):
    if NDCG_list_e[i] > NDCG_list_p[i]:
        d_NDCG.append(NDCG_list_e[i] - NDCG_list_p[i])
        d_NDCG_ix.append(i)
print(len(d_NDCG), len(d_NDCG_ix))

### [Based on Lecture 2]
__Step 4: Implement Interleaving (15 points).__

Implement ~~2~~ 1 interleaving algorithms: (1) Team-Draft Interleaving OR Balanced Interleaving, ~~AND (2), Probabilistic Interleaving.~~ The interleaving algorithms should (a) given two rankings of relevance interleave them into a single ranking, and (b) given the users clicks on the interleaved ranking assign credit to the algorithms that produced the rankings.

(Note 4: Note here that as opposed to a normal interleaving experiment where rankings consists of urls or docids, in our case the rankings consist of relevance labels. Hence in this case (a) you will assume that E and P return different documents, (b) the interleaved ranking will also be a ranking of labels.)


In [ ]:
def Interleaving(rank_p, rank_e):
    toincoss = np.round(np.random.rand())
    interleaved_rank = []
    for i in range(len(rank_p)):
        if i % 2 == toincoss:
            interleaved_rank.append(rank_p[i])
        else:
            interleaved_rank.append(rank_e[i])
    return interleaved_rank
            